In [1]:
library(dplyr)
library(tidyr)
library(pheatmap)
library(viridis)
library(tools)
library(cluster)
library(igraph)
library(tibble)

library(clusterProfiler)
library(org.Hs.eg.db)
library(dplyr)

Warning message:
“程辑包‘dplyr’是用R版本4.2.3 来建造的”

载入程辑包：‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“程辑包‘tidyr’是用R版本4.2.3 来建造的”
Warning message:
“程辑包‘viridis’是用R版本4.2.3 来建造的”
载入需要的程辑包：viridisLite

Warning message:
“程辑包‘viridisLite’是用R版本4.2.3 来建造的”
Warning message:
“程辑包‘cluster’是用R版本4.2.3 来建造的”

载入程辑包：‘igraph’


The following object is masked from ‘package:tidyr’:

    crossing


The following objects are masked from ‘package:dplyr’:

    as_data_frame, groups, union


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:

    union



载入程辑包：‘tibble’


The following object is masked from ‘package:igraph’:

    as_data_frame


Warning message:
“程辑包‘clusterProfiler’是用R版本4.2.2 来建造的”


Registered S3 methods overwritten by 'treeio':
  method              from    
  

In [2]:
# 读入
signature_CD8T <- read.csv(
  "3.cGEP_topgene/ALL_cGEP_top_genes_long_wide_by_cGEP1_70.csv",
  check.names = FALSE,
  stringsAsFactors = FALSE)
# 转成 list（核心）
cluster_genes <- lapply(signature_CD8T, function(x) {
  x[!is.na(x) & x != ""]
})


gene_df <- data.frame(
  cGEP_Cluster = names(cluster_genes),  # 确保这里是 cGEP1, cGEP2...
  cGEP_signature = sapply(cluster_genes, function(x) paste(x, collapse = ",")),
  stringsAsFactors = FALSE
)
gene_df

,cGEP_Cluster,cGEP_signature
,<chr>,<chr>
cGEP1,cGEP1,"TotalSeqC0953,RP11-864N7.2,RP11-234A1.1,RP11-425L10.1,RPL13P12,RPS12,TotalseqC0251,RPL13,RPL32,AC024293.1,EEF1A1,RPL34,AC016739.2,RPS18,RPS28,RPS3A,RPL18AP3,RPL39,RPL10,RPL11,RPS14,RPS8,RPL30,RPS23,RPS6,RPS13,RPL19,RPL37,RPL18A,RPS27"
cGEP2,cGEP2,"ISG15,MX1,IFI6,IFIT3,IFIT1,RSAD2,OAS1,IFI44L,ISG20,LY6E,IFITM1,CMPK2,HERC5,IRF7,OAS3,MX2,EPSTI1,PLSCR1,STAT1,MT2A,USP18,IFIT2,IFI35,EIF2AK2,XAF1,IFI44,SAMD9L,TNFSF10,OASL,LAMP3"
cGEP3,cGEP3,"CYTB,ATP6,COX3,ND4L,COX2,ND1,ND5,ND2,ATP8,RPS14P3,ND3,MT-ND1,GAS5,MT-CO3,MT-CO2,ND4,HLA-H,MT-ND2,MT-ND3,MT-ATP6,LOC100507217,MT-CYB,MT-ND4,MT-ND4L,MT-ND5,MT-CO1,ND6,MT-RNR2,AC026366.1,CRISP3"
cGEP4,cGEP4,"RPL13P12,TotalSeqC0952,EEF1A1,TotalSeqC0955,RPL13,RPL34,TPT1,RPL32,RPS12,AC010343.1,RPLP1,AL365357.1,IL7R,RPS4XP22,RPS8,AC016739.1,RPS6,RPL10,RPL11,RPS4X,RPS18,AL590867.2,RPS14,RPS13,CCR7,FTH1,RPS3A,RPL30,RPL3,RPL19"
cGEP5,cGEP5,"MT-ND4,MT-CO2,MT-CO3,MT-ATP6,MT-CYB,MT-CO1,MT-ND2,MT-ND1,MT-ND3,MT-ND5,SLIT1,PVRL4,MT-ND4L,MTATP6P1,MT-RNR1,TMPRSS11F,ANKRD30A,MTRNR2L10,RFX6,MT-TH,MT-RNR2,WARS2-IT1,PABPC4L,KCNN2,RNF225,SCGB1D2,MTRNR2L9,TMSB4XP8,MT-ATP8,MTRNR2L8"
cGEP6,cGEP6,"GAPDH,KRT86,GZMB,ENTPD1,TNFRSF9,TIGIT,TNFRSF18,CD7,LAYN,AC092580.4,LOC101929531,SNX9,ALOX5AP,TPI1,CXCL13,TNS3,PGAM1,PKM,HAVCR2,AC002331.1,RBPJ,PDLIM4,LAG3,CTSD,ITGAE,CTLA4,GOLIM4,IFITM10,MYO1E,CD82"
cGEP7,cGEP7,"RP11-1033A18.1,DNAJB1,HSP90AA1,HSPA1A,RP11-485G4.2,HSPA1B,HSPH1,HSP90AB1,HSPE1,HSPD1,HSPA8,DNAJA1,RP11-153M3.1,CACYBP,HSP90AA2P,GOT2P2,UBB,HSPB1,BAG3,ZFAND2A,TIMM8AP1,HSPA6,GGNBP1,PPP1R15A,UBC,RP11-142L4.2,HSP90AB3P,MIRLET7D,SERPINH1,LOC101928304"
cGEP8,cGEP8,"GNLY,S100A4,TMSB10,TRBV7-6,IFITM2,RPL39,S100A6,TRAV8-1,ATP5E,SH3BGRL3,SERF2,FXYD5,MT-ND3,RPL41,MYL12A,S100A11,TMSB4X,RPS4Y1,GZMB,RPL36A,TMA7,HSPA1A,RPS18,OST4,RPS21,S100A10,TRAV10,HCST,PFN1,IFITM1"
cGEP9,cGEP9,"HBB,HBD,AHSP,SNCA,CA1,HBA2,ALAS2,HBA1,SLC4A1,SELENBP1,SLC25A37,PDZK1IP1,TRIM58,HBM,HBQ1,GMPR,HEMGN,KLHL33,KLF1,SLC25A39,HBG2,STRADB,EPB42,PLEK2,BLVRB,NFIX,NFE2,TNS1,BNIP3L,IGF2BP2"


In [3]:
#CD8T_gep_anno <- read.csv('./3.cGEP_topgene/3.3.CD8T_GEP70_Anno_final.csv')
CD8T_gep_anno <- read.csv('/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.6.cNMF_menchmark/cGEP_signature/CD8T_cGEP70.csv')
CD8T_gep_anno

cGEP_Cluster,cGEP_Anno_Name,Category
<chr>,<chr>,<chr>
cGEP1,Ribo_Bio,Functional
cGEP10,MAIT,Lineage
cGEP11,Activ_IEG,Functional
cGEP12,Cyto_Synapse,Functional
cGEP13,Myeloid_Inflam,Doublet
cGEP14,Bcell_APC,Doublet
cGEP15,Transl_Protein,Functional
cGEP16,CD8-Trm_Metab,Lineage
cGEP17,CD8_GNLY_NKlike,Lineage


In [4]:
final_combined_df <- CD8T_gep_anno %>%
  left_join(gene_df, by = "cGEP_Cluster")
final_combined_df

cGEP_Cluster,cGEP_Anno_Name,Category,cGEP_signature
<chr>,<chr>,<chr>,<chr>
cGEP1,Ribo_Bio,Functional,"TotalSeqC0953,RP11-864N7.2,RP11-234A1.1,RP11-425L10.1,RPL13P12,RPS12,TotalseqC0251,RPL13,RPL32,AC024293.1,EEF1A1,RPL34,AC016739.2,RPS18,RPS28,RPS3A,RPL18AP3,RPL39,RPL10,RPL11,RPS14,RPS8,RPL30,RPS23,RPS6,RPS13,RPL19,RPL37,RPL18A,RPS27"
cGEP10,MAIT,Lineage,"KLRB1,SLC4A10,NCR3,AMICA1,CEBPD,IL7R,LTB,S100A4,DUSP1,CCR6,LTK,ZBTB16,AQP3,NFKBIA,IL23R,IL4I1,TSPAN15,RP11-693N9.2,ME1,RORC,LST1,CD69,RORA,CTD-3014M21.1,IL18RAP,IL22,CCL20,IL26,CTSH,PHACTR2"
cGEP11,Activ_IEG,Functional,"FOS,JUN,DUSP1,FOSB,COX1,KLF6,CD69,JUNB,ZFP36,TSC22D3,ND4,TNFAIP3,NFKBIA,PPP1R15A,NR4A2,CYTB,RNR2,COX2,ND4L,IER2,ZFP36L2,COX3,CXCR4,BTG2,ATP6,H3F3B,ND1,EGR1,CCL4,NR4A1"
cGEP12,Cyto_Synapse,Functional,"TotalSeqC0954,LOC100996919,NKG7,GZMH,TotalseqC0252,GZMA,CCL5,ACTB,PFN1,SH3BGRL3,TMSB4X,LOC100996809,CFL1,LOC101060038,CORO1A,PLAAT4,HCST,HLA-DPA1,CLIC1,ACTG1,HLA-DRB1,HLA-DPB1,B2M,HLA-C,CD8A,GZMB,IL32,LOC102723461,CST7,CD3D"
cGEP13,Myeloid_Inflam,Doublet,"LYZ,LINC01272,S100A9,LOC100130520,FCN1,CST3,S100A8,SERPINA1,SPI1,TYROBP,CSTA,VCAN,CYBB,FPR1,CD14,LOC100507639,MAFB,TNFAIP2,MNDA,CLEC7A,TMEM176B,RP11-1143G9.4,LOC102723715,CXCL8,TNFSF13,SLC7A7,LILRB2,CSF3R,LILRB4,RBP7"
cGEP14,Bcell_APC,Doublet,"HLA-DRA,CD74,CD79A,AL096701.2,MS4A1,VPREB1,BANK1,MEF2C,TCL1A,VPREB3,HLA-DPA1,CD22,HNRNPA1P32,HLA-DRB1,HLA-DQA1,HLA-DQB1,HMGB1P13,CD79B,NAPSB,MARCH1,AL121821.2,HLA-DPB1,HLA-DRB6,ARHGAP24,SPIB,CD19,RALGPS2,CD24,SWAP70,FCER2"
cGEP15,Transl_Protein,Functional,"RP11.295G20.2,CPEB1.AS1,RPL27A,CTD.2012K14.8,RPL31,RPS17,RPL13A,SCGB3A2,RPS29,RPL21,SGMS1.AS1,SFTPC,RP11.378A13.2,RP11-347P5.1,RPS20,RPLP2,RPL7,CCL18,RPS10,RPS27,MALAT1,ATP5EP2,RPL26,TMSB4X,RPL23A,RPL41,FABP4,MARCO,RPL36A,RP5.1042K10.10"
cGEP16,CD8-Trm_Metab,Lineage,"ONECUT2,DMBT1,KRT20,HRCT1,ALDOB,OLFM4,MEP1A,FABP2,EPCAM,TRIM54,TRAV9-2,CD52,HLA-C,A1CF,FAM3D,S100A14,AKR7A3,SI,CLRN3,B2M,CX3CR1,HAAO,S100A4,TM6SF2,GDF15,PART1,CCL25,TRAC,TBX3,GP2"
cGEP17,CD8_GNLY_NKlike,Lineage,"RP11-354E11.2,CRISPLD1,NTM,HDC,GNLY,TYROBP,XCL1,XCL2,FCER1G,SPTSSB,KLRC1,ALKAL2,RP11-245G13.2,CPA3,CTD-2165H16.3,PPM1H,TPSB2,TNFRSF18,KLRF1,IFITM3,KIR2DL4,IL2RB,SELL,IGFBP7,CTSW,NCAM1,CAPG,FLJ21408,DACH1,CNRIP1"


In [5]:
#OUT_FILE <- "./3.cGEP_topgene/3.3.CD8T_cGEP_Anno_Complete_With_Genes_filt.csv"
OUT_FILE <- '/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.6.cNMF_menchmark/cGEP_signature/CD8T_cGEP70_signature.csv'
write.csv(final_combined_df, OUT_FILE, row.names = FALSE)